# 🏗️ Production AI Engineering Lab
### Three foundational pillars: Prompt Architecture · State Management · Model Routing

> *'Most AI failures are system design failures. The model is generating probabilistic outputs;
> the engineering around it determines if those outputs are usable.'*

| Pillar | Core idea |
|--------|----------|
| 1. Prompt Architecture | Prompts as versioned, testable, schema-enforced interfaces |
| 2. State Management | Immutable logs, ownership boundaries, replayable decisions |
| 3. Model Routing | Right model for the job; switching is config, not rewrite |

> Set your `OPENAI_API_KEY` in Section 0 before running.

## Section 0 — Setup

In [46]:
!pip install openai pydantic tiktoken -q

import os, json, uuid, hashlib, time, re
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Optional, Any
from enum import Enum
from pydantic import BaseModel, Field, ValidationError

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
client = OpenAI()

def call_openai(prompt: str, system: str = '', model: str = 'gpt-4o-mini',
                max_tokens: int = 512, temperature: float = 0.2) -> tuple[str, int, int]:
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': prompt})
    r = client.chat.completions.create(
        model=model, messages=msgs,
        max_tokens=max_tokens, temperature=temperature
    )
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens

print('Setup complete. Models available: gpt-4o-mini, gpt-4o')


zsh:1: command not found: pip
Setup complete. Models available: gpt-4o-mini, gpt-4o


---
# Pillar 1 — Prompt Architecture
**Treat prompts as deterministic reasoning interfaces, not magic strings.**

Three practices:
1. **Versioning** — every prompt has an ID, version, and changelog. Roll back like code.
2. **Output schemas** — enforce structured output with Pydantic. No free-form parsing.
3. **Testability** — evaluate prompts against objective expectations. Red/green tests, not vibes.

## 1a — Prompt Versioning

In [ ]:
@dataclass
class PromptVersion:
    prompt_id:   str
    version:     str           # semver: major.minor.patch
    system:      str
    user_template: str         # use {variable} placeholders
    changelog:   str = ''
    created_at:  str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    @property
    def fingerprint(self) -> str:
        raw = self.system + self.user_template
        return hashlib.sha256(raw.encode()).hexdigest()[:10]

    def render(self, **kwargs) -> str:
        return self.user_template.format(**kwargs)


class PromptRegistry:
    def __init__(self):
        self._store: dict[str, list[PromptVersion]] = {}

    def register(self, pv: PromptVersion):
        self._store.setdefault(pv.prompt_id, []).append(pv)

    def get(self, prompt_id: str, version: str = 'latest') -> PromptVersion:
        versions = self._store.get(prompt_id, [])
        if not versions:
            raise KeyError(f'Prompt {prompt_id!r} not found')
        if version == 'latest':
            return versions[-1]
        match = [v for v in versions if v.version == version]
        if not match:
            raise KeyError(f'Version {version} not found for {prompt_id}')
        return match[0]

    def history(self, prompt_id: str):
        for pv in self._store.get(prompt_id, []):
            print(f'  v{pv.version}  [{pv.fingerprint}]  {pv.changelog or "(no changelog)"}')


registry = PromptRegistry()

# Register two versions of the same prompt
registry.register(PromptVersion(
    prompt_id='classify_ticket',
    version='1.0.0',
    system='You are a support ticket classifier.',
    user_template='Classify this ticket: {ticket_text}\nReturn one of: billing, technical, general',
    changelog='initial version'
))

registry.register(PromptVersion(
    prompt_id='classify_ticket',
    version='1.1.0',
    system='You are a precise support ticket classifier. Always output valid JSON.',
    user_template=(
        'Classify this support ticket and return JSON only.\n'
        'Schema: {{"category": str, "urgency": "low"|"medium"|"high", "confidence": 0.0-1.0}}\n\n'
        'Ticket: {ticket_text}'
    ),
    changelog='added urgency + confidence fields; enforced JSON output'
))

registry.register(PromptVersion(
    prompt_id='classify_ticket',
    version='1.2.0',
    system=(
        'You are a precise support ticket classifier.\n'
        'Classify tickets accurately into categories and assess urgency.\n'
        'Always return valid JSON. Do not include any extra text.'
    ),
    user_template=(
        'Classify the following support ticket.\n\n'
        'Allowed categories: "billing", "technical", "general"\n\n'
        'Return JSON only using this schema:\n'
        'Schema: {{"category": "billing" | "technical" | "general", "urgency": "low" | "medium" | "high", "confidence": float (0.0 to 1.0)}}\n\n'
        'Rules:\n'
        '- category must be one of the allowed values\n'
        '- urgency reflects how critical the issue is\n'
        '- confidence reflects classification certainty\n'
        '- output strictly valid JSON (no markdown, no explanation)\n\n'
        'Ticket: {ticket_text}'
    ),
    changelog=(
        'consolidated v1.0 and v1.1; '
        'restricted category to original enum; '
        'retained JSON schema with urgency and confidence; '
        'added stricter output rules for reliability'
    )
))

print('Prompt history for classify_ticket:')
registry.history('classify_ticket')

# Roll back to v1.0.0
pv_old = registry.get('classify_ticket', version='1.0.0')
pv_new = registry.get('classify_ticket', version='latest')
print(f'\nLatest fingerprint : {pv_new.fingerprint}')
print(f'v1.0.0 fingerprint : {pv_old.fingerprint}')
print('Fingerprints differ (prompts changed):', pv_old.fingerprint != pv_new.fingerprint)
print('latest ',  registry.get('classify_ticket', version='latest'))


Prompt history for classify_ticket:
  v1.0.0  [bec0e320e5]  initial version
  v1.1.0  [f8586a69a1]  added urgency + confidence fields; enforced JSON output
  v1.2.0  [67fe2f571d]  consolidated v1.0 and v1.1; restricted category to original enum; retained JSON schema with urgency and confidence; added stricter output rules for reliability

Latest fingerprint : 67fe2f571d
v1.0.0 fingerprint : bec0e320e5
Fingerprints differ (prompts changed): True
latest  PromptVersion(prompt_id='classify_ticket', version='1.2.0', system='You are a precise support ticket classifier.\nClassify tickets accurately into categories and assess urgency.\nAlways return valid JSON. Do not include any extra text.', user_template='Classify the following support ticket.\n\nAllowed categories: "billing", "technical", "general"\n\nReturn JSON only using this schema:\nSchema: {{"category": "billing" | "technical" | "general", "urgency": "low" | "medium" | "high", "confidence": float (0.0 to 1.0)}}\n\nßßßßRules:\n- categ

## 1b — Output Schemas (Pydantic enforcement)

In [21]:
class TicketCategory(str, Enum):
    BILLING   = 'billing'
    TECHNICAL = 'technical'
    GENERAL   = 'general'

class Urgency(str, Enum):
    LOW    = 'low'
    MEDIUM = 'medium'
    HIGH   = 'high'

class TicketClassification(BaseModel):
    category:   TicketCategory
    urgency:    Urgency
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning:  Optional[str] = None


def classify_ticket(ticket_text: str, prompt_version: str = 'latest') -> TicketClassification:
    pv = registry.get('classify_ticket', version=prompt_version)
    prompt = pv.render(ticket_text=ticket_text)
    raw, in_toks, out_toks = call_openai(prompt, system=pv.system)

    # Strip markdown fences if model adds them
    clean = re.sub(r'```json\s*|```', '', raw).strip()
    try:
        data   = json.loads(clean)
        result = TicketClassification(**data)
        return result
    except (json.JSONDecodeError, ValidationError) as e:
        raise ValueError(f'Schema validation failed: {e}\nRaw output: {raw[:200]}')


# Test tickets
tickets = [
    'I was charged twice for my subscription this month!',
    'The API is returning 500 errors on /v2/embeddings since 2pm.',
    'Hi, just wondering what your refund policy is.',
    'Hi Can you explain how your refund policy works?',
    'hi Can you explain how your refund policy works? I was charged twice for my subscription this month!',
    'hi Can you give me link of API Docs? The API is returning 500 errors on /v2/embeddings since 2pm.',
    'hi Can you explain how your refund policy works? The API is returning 500 errors on /v2/embeddings since 2pm. I was charged twice for my subscription this month!'
]

print('=== Output Schema Enforcement Demo ===\n')
for t in tickets:
    result = classify_ticket(t, prompt_version='latest')
    print("result", result)
    print(f'Ticket : {t[:600]}')
    print(f'Result : category={result.category.value}  urgency={result.urgency.value}'
          f'  confidence={result.confidence:.2f}')
    print()


=== Output Schema Enforcement Demo ===

result category=<TicketCategory.BILLING: 'billing'> urgency=<Urgency.HIGH: 'high'> confidence=0.95 reasoning=None
Ticket : I was charged twice for my subscription this month!
Result : category=billing  urgency=high  confidence=0.95

result category=<TicketCategory.TECHNICAL: 'technical'> urgency=<Urgency.HIGH: 'high'> confidence=0.95 reasoning=None
Ticket : The API is returning 500 errors on /v2/embeddings since 2pm.
Result : category=technical  urgency=high  confidence=0.95

result category=<TicketCategory.BILLING: 'billing'> urgency=<Urgency.MEDIUM: 'medium'> confidence=0.85 reasoning=None
Ticket : Hi, just wondering what your refund policy is.
Result : category=billing  urgency=medium  confidence=0.85

result category=<TicketCategory.BILLING: 'billing'> urgency=<Urgency.MEDIUM: 'medium'> confidence=0.85 reasoning=None
Ticket : Hi Can you explain how your refund policy works?
Result : category=billing  urgency=medium  confidence=0.85

result ca

## 1c — Prompt Testability

In [27]:
@dataclass
class PromptTestCase:
    input:           str
    expected_category: str
    expected_urgency:  str
    min_confidence:    float = 0.7

@dataclass
class TestResult:
    passed:   bool
    input:    str
    expected: dict
    actual:   dict
    reason:   str = ''


def run_prompt_tests(prompt_version: str,
                     test_cases: list[PromptTestCase]) -> list[TestResult]:
    results = []
    for tc in test_cases:
        try:
            out = classify_ticket(tc.input, prompt_version=prompt_version)
            checks = [
                (out.category.value.lower() == tc.expected_category.lower(),
                f'category: got {out.category.value!r}, expected {tc.expected_category!r}'),

                (out.urgency.value.lower() == tc.expected_urgency.lower(),
                f'urgency: got {out.urgency.value!r}, expected {tc.expected_urgency!r}'),

                (out.confidence >= tc.min_confidence,
                f'confidence {out.confidence:.2f} < min {tc.min_confidence}'),
            ]
            failed = [msg for ok, msg in checks if not ok]
            results.append(TestResult(
                passed   = len(failed) == 0,
                input    = tc.input,
                expected = {'category': tc.expected_category, 'urgency': tc.expected_urgency},
                actual   = {'category': out.category.value,   'urgency': out.urgency.value,
                            'confidence': out.confidence},
                reason   = '; '.join(failed)
            ))
        except Exception as ex:
            print("=========== ", str(ex))
            results.append(TestResult(
                passed=False, input=tc.input,
                expected={}, actual={}, reason=str(ex)
            ))
    return results


def print_test_report(version: str, results: list[TestResult]):
    passed = sum(r.passed for r in results)
    print(f'\nPrompt v{version} — {passed}/{len(results)} tests passed')
    print('-' * 50)
    for r in results:
        icon = '✅' if r.passed else '❌'
        print(f'{icon} {r.input[:55]}')
        if not r.passed:
            print(f'   FAIL: {r}')
        else:
            print(f'   category={r.actual["category"]}  '
                  f'urgency={r.actual["urgency"]}  '
                  f'confidence={r.actual["confidence"]:.2f}')


TEST_SUITE = [
    PromptTestCase('Charged twice for subscription this month!',
                   expected_category='billing',   expected_urgency='high'),
    PromptTestCase('API returning 500 errors on /v2/embeddings.',
                   expected_category='technical', expected_urgency='high'),
    PromptTestCase('What is your refund policy?',
                   expected_category='billing',   expected_urgency='low'),
    PromptTestCase('App crashes when I upload files over 10MB.',
                   expected_category='technical', expected_urgency='medium'),
    PromptTestCase('Just wanted to say your product is great!',
                   expected_category='general',   expected_urgency='low'),
    PromptTestCase('When is the next update!',
                   expected_category='general',   expected_urgency='low'),
    PromptTestCase('When can i expect my refund?',
                   expected_category='billing',   expected_urgency='medium'),
]


print('=== Prompt Test Suite ===\n')
# Compare v1.0.0 (no schema) vs v1.1.0 (schema-enforced)
# Note: v1.0.0 will likely fail schema validation - that is the point
print('Testing v1.2.0 (schema-enforced):')
results_new = run_prompt_tests('1.2.0', TEST_SUITE)
print_test_report('1.2.0', results_new)

total  = len(results_new)
passed = sum(r.passed for r in results_new)
print(f'\nPass rate: {passed/total*100:.0f}% -- if this drops after a prompt change, roll back')


=== Prompt Test Suite ===

Testing v1.2.0 (schema-enforced):

Prompt v1.2.0 — 4/7 tests passed
--------------------------------------------------
✅ Charged twice for subscription this month!
   category=billing  urgency=high  confidence=0.95
✅ API returning 500 errors on /v2/embeddings.
   category=technical  urgency=high  confidence=0.95
❌ What is your refund policy?
   FAIL: TestResult(passed=False, input='What is your refund policy?', expected={'category': 'billing', 'urgency': 'low'}, actual={'category': 'billing', 'urgency': 'medium', 'confidence': 0.85}, reason="urgency: got 'medium', expected 'low'")
❌ App crashes when I upload files over 10MB.
   FAIL: TestResult(passed=False, input='App crashes when I upload files over 10MB.', expected={'category': 'technical', 'urgency': 'medium'}, actual={'category': 'technical', 'urgency': 'high', 'confidence': 0.95}, reason="urgency: got 'high', expected 'medium'")
✅ Just wanted to say your product is great!
   category=general  urgency=lo

---
# Pillar 2 — State Management
**The primary failure point for multi-agent systems.**

Three practices:
1. **Ownership boundaries** — one service owns one slice of state. No shared mutable globals.
2. **Immutable event log** — every decision is recorded. Replay any point in time.
3. **Bounded contexts** — agents read/write through an interface, never directly to shared memory.

## 2a — Immutable Event Log + Replayability

In [28]:
class EventType(str, Enum):
    TICKET_RECEIVED   = 'ticket_received'
    TICKET_CLASSIFIED = 'ticket_classified'
    AGENT_ASSIGNED    = 'agent_assigned'
    RESPONSE_DRAFTED  = 'response_drafted'
    TICKET_RESOLVED   = 'ticket_resolved'


@dataclass(frozen=True)   # frozen = immutable after creation
class Event:
    event_id:   str
    event_type: EventType
    entity_id:  str          # ticket_id, user_id, etc.
    payload:    dict
    ts:         str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    caused_by:  Optional[str] = None   # event_id that triggered this one

    def __post_init__(self):
        # Enforce immutability of payload too
        object.__setattr__(self, 'payload', dict(self.payload))


class ImmutableEventLog:
    def __init__(self):
        self._log: list[Event] = []

    def append(self, event: Event):
        self._log.append(event)
        print(f'  LOG [{event.event_type.value:25s}] entity={event.entity_id}'
              f'  payload={json.dumps(event.payload)[:60]}')

    def events_for(self, entity_id: str) -> list[Event]:
        return [e for e in self._log if e.entity_id == entity_id]

    def replay(self, entity_id: str) -> dict:
        state: dict = {}
        for event in self.events_for(entity_id):
            if event.event_type == EventType.TICKET_RECEIVED:
                state.update({'status': 'open', 'text': event.payload.get('text')})
            elif event.event_type == EventType.TICKET_CLASSIFIED:
                state.update({'category': event.payload.get('category'),
                              'urgency':  event.payload.get('urgency')})
            elif event.event_type == EventType.AGENT_ASSIGNED:
                state['assigned_to'] = event.payload.get('agent')
            elif event.event_type == EventType.RESPONSE_DRAFTED:
                state['draft'] = event.payload.get('draft')
            elif event.event_type == EventType.TICKET_RESOLVED:
                state['status'] = 'resolved'
        return state

    def audit_trail(self, entity_id: str):
        print(f'\nAudit trail for {entity_id}:')
        for e in self.events_for(entity_id):
            caused = f' (caused by {e.caused_by})' if e.caused_by else ''
            print(f'  {e.ts}  {e.event_type.value}{caused}')


event_log = ImmutableEventLog()

print('=== Immutable Event Log Demo ===\n')

# Simulate a ticket lifecycle
ticket_id = f'TKT-{uuid.uuid4().hex[:6].upper()}'
ticket_text = 'API returning 500 errors on /v2/embeddings since 2pm.'

# 1. Ticket received
e1 = Event(event_id=uuid.uuid4().hex[:8], event_type=EventType.TICKET_RECEIVED,
           entity_id=ticket_id, payload={'text': ticket_text, 'source': 'email'})
event_log.append(e1)

# 2. Classify with prompt
classification = classify_ticket(ticket_text, prompt_version='1.2.0')
e2 = Event(event_id=uuid.uuid4().hex[:8], event_type=EventType.TICKET_CLASSIFIED,
           entity_id=ticket_id, caused_by=e1.event_id,
           payload={'category': classification.category.value,
                    'urgency':  classification.urgency.value,
                    'confidence': classification.confidence})
event_log.append(e2)

# 3. Assign agent based on category
agent_map = {'technical': 'eng-team', 'billing': 'finance-team', 'general': 'support-team'}
assigned  = agent_map[classification.category.value]
e3 = Event(event_id=uuid.uuid4().hex[:8], event_type=EventType.AGENT_ASSIGNED,
           entity_id=ticket_id, caused_by=e2.event_id,
           payload={'agent': assigned})
event_log.append(e3)

# 4. Replay state from events
print('\nReplayed state from event log:')
state = event_log.replay(ticket_id)
print(json.dumps(state, indent=2))

event_log.audit_trail(ticket_id)


=== Immutable Event Log Demo ===

  LOG [ticket_received          ] entity=TKT-61E8F2  payload={"text": "API returning 500 errors on /v2/embeddings since 2
  LOG [ticket_classified        ] entity=TKT-61E8F2  payload={"category": "technical", "urgency": "high", "confidence": 0
  LOG [agent_assigned           ] entity=TKT-61E8F2  payload={"agent": "eng-team"}

Replayed state from event log:
{
  "status": "open",
  "text": "API returning 500 errors on /v2/embeddings since 2pm.",
  "category": "technical",
  "urgency": "high",
  "assigned_to": "eng-team"
}

Audit trail for TKT-61E8F2:
  2026-03-22T07:20:03.955447+00:00  ticket_received
  2026-03-22T07:20:04.841430+00:00  ticket_classified (caused by 30d973a5)
  2026-03-22T07:20:04.842020+00:00  agent_assigned (caused by 9a1cbd5e)


## 2b — Ownership Boundaries & Bounded Contexts

In [29]:
# Each BoundedContext owns exactly one slice of state.
# Agents never touch each other's state directly -- they emit events.

class TicketOwnershipError(Exception):
    pass


class BoundedContext:
    def __init__(self, name: str, owned_fields: set[str], event_log: ImmutableEventLog):
        self.name          = name
        self.owned_fields  = owned_fields
        self._state: dict  = {}
        self._log          = event_log

    def write(self, entity_id: str, updates: dict, event_type: EventType,
              caused_by: Optional[str] = None):
        # Enforce ownership: only write to fields this context owns
        illegal = set(updates.keys()) - self.owned_fields
        if illegal:
            raise TicketOwnershipError(
                f'Context {self.name!r} cannot write fields {illegal}. '
                f'Owned: {self.owned_fields}'
            )
        self._state.setdefault(entity_id, {}).update(updates)
        ev = Event(event_id=uuid.uuid4().hex[:8], event_type=event_type,
                   entity_id=entity_id, payload=updates, caused_by=caused_by)
        self._log.append(ev)
        return ev

    def read(self, entity_id: str) -> dict:
        return self._state.get(entity_id, {})


# Define ownership slices
classification_ctx = BoundedContext(
    'ClassificationService',
    owned_fields={'category', 'urgency', 'confidence'},
    event_log=event_log
)
assignment_ctx = BoundedContext(
    'AssignmentService',
    owned_fields={'assigned_to', 'assigned_at'},
    event_log=event_log
)
resolution_ctx = BoundedContext(
    'ResolutionService',
    owned_fields={'status', 'resolved_at', 'resolution_note'},
    event_log=event_log
)

print('=== Ownership Boundaries Demo ===\n')
tid = f'TKT-{uuid.uuid4().hex[:6].upper()}'

# Each context writes only what it owns
classification_ctx.write(tid,
    {'category': 'technical', 'urgency': 'high', 'confidence': 0.92},
    EventType.TICKET_CLASSIFIED)

assignment_ctx.write(tid,
    {'assigned_to': 'eng-team', 'assigned_at': datetime.now(timezone.utc).isoformat()},
    EventType.AGENT_ASSIGNED)

resolution_ctx.write(tid,
    {'status': 'resolved', 'resolved_at': datetime.now(timezone.utc).isoformat(),
     'resolution_note': 'Fixed upstream rate limiter.'},
    EventType.TICKET_RESOLVED)

print('\nCurrent state per context:')
for ctx in [classification_ctx, assignment_ctx, resolution_ctx]:
    print(f'  [{ctx.name}]: {ctx.read(tid)}')

# Now show enforcement: AssignmentService tries to write a field it does not own
print('\nEnforcing ownership boundary (AssignmentService tries to set "category"):')
try:
    assignment_ctx.write(tid, {'category': 'billing'}, EventType.TICKET_CLASSIFIED)
except TicketOwnershipError as e:
    print(f'  OwnershipError caught: {e}')
    print('  Boundary held. ✅')


=== Ownership Boundaries Demo ===

  LOG [ticket_classified        ] entity=TKT-6B55B5  payload={"category": "technical", "urgency": "high", "confidence": 0
  LOG [agent_assigned           ] entity=TKT-6B55B5  payload={"assigned_to": "eng-team", "assigned_at": "2026-03-22T07:37
  LOG [ticket_resolved          ] entity=TKT-6B55B5  payload={"status": "resolved", "resolved_at": "2026-03-22T07:37:12.3

Current state per context:
  [ClassificationService]: {'category': 'technical', 'urgency': 'high', 'confidence': 0.92}
  [AssignmentService]: {'assigned_to': 'eng-team', 'assigned_at': '2026-03-22T07:37:12.381285+00:00'}
  [ResolutionService]: {'status': 'resolved', 'resolved_at': '2026-03-22T07:37:12.381346+00:00', 'resolution_note': 'Fixed upstream rate limiter.'}

Enforcing ownership boundary (AssignmentService tries to set "category"):
  OwnershipError caught: Context 'AssignmentService' cannot write fields {'category'}. Owned: {'assigned_at', 'assigned_to'}
  Boundary held. ✅


---
# Pillar 3 — Model Selection & Routing
**Model choice is a distraction from architecture. Design for interchangeability.**

Three practices:
1. **Dynamic routing** — small fast models for classification/extraction; frontier models for reasoning.
2. **Contextual benchmarking** — measure on *your* pipeline, not global leaderboards.
3. **Interchangeability** — switching models is a config change, not a rewrite.

## 3a — Dynamic Model Router

In [47]:
import time
from dataclasses import dataclass
from enum import Enum


# -----------------------------
# Complexity Enum
# -----------------------------
class TaskComplexity(str, Enum):
    SIMPLE  = 'simple'
    MEDIUM  = 'medium'
    COMPLEX = 'complex'


# -----------------------------
# Model Config
# -----------------------------
@dataclass
class ModelConfig:
    model_id: str
    max_tokens: int = 512
    temperature: float = 0.2
    cost_per_1m_in: float = 0.0
    cost_per_1m_out: float = 0.0

    def cost(self, in_toks: int, out_toks: int) -> float:
        return (in_toks / 1e6) * self.cost_per_1m_in + \
               (out_toks / 1e6) * self.cost_per_1m_out


# -----------------------------
# Model Registry
# -----------------------------
MODEL_REGISTRY = {
    TaskComplexity.SIMPLE: ModelConfig(
        'gpt-4o-mini', max_tokens=256, temperature=0.0,
        cost_per_1m_in=0.15, cost_per_1m_out=0.60
    ),
    TaskComplexity.MEDIUM: ModelConfig(
        'gpt-4o-mini', max_tokens=512, temperature=0.1,
        cost_per_1m_in=0.15, cost_per_1m_out=0.60
    ),
    TaskComplexity.COMPLEX: ModelConfig(
        'gpt-4o', max_tokens=1024, temperature=0.2,
        cost_per_1m_in=2.50, cost_per_1m_out=10.00
    ),
}


# -----------------------------
# Complexity Classifier
# -----------------------------
def classify_complexity(task_description: str) -> TaskComplexity:
    prompt = (
        'Classify the complexity of this AI task into exactly one word: '
        'simple, medium, or complex.\n\n'
        'Rules:\n'
        '- simple: classification, extraction, yes/no\n'
        '- medium: summarisation, structured output\n'
        '- complex: reasoning, code, deep analysis\n\n'
        f'Task: {task_description}\n\n'
        'Answer (one word only):'
    )

    raw, _, _ = call_openai(prompt, model='gpt-4o-mini',
                            max_tokens=5, temperature=0.0)

    word = raw.strip().lower()

    for c in TaskComplexity:
        if c.value in word:
            return c

    return TaskComplexity.MEDIUM  # safe fallback


# -----------------------------
# Routed Call
# -----------------------------
def routed_call(task_description: str, prompt: str, system: str = ''):
    complexity = classify_complexity(task_description)
    cfg = MODEL_REGISTRY[complexity]

    t0 = time.time()

    response, in_toks, out_toks = call_openai(
        prompt,
        system=system,
        model=cfg.model_id,
        max_tokens=cfg.max_tokens,
        temperature=cfg.temperature
    )

    latency = time.time() - t0
    cost = cfg.cost(in_toks, out_toks)

    return {
        'task': task_description,
        'complexity': complexity,
        'model': cfg,
        'response': response,
        'in_tokens': in_toks,
        'out_tokens': out_toks,
        'latency_s': round(latency, 2),
        'cost_usd': cost
    }


# -----------------------------
# Correct Cost Comparison
# -----------------------------
def compute_all_gpt4o_cost(results):
    cfg = MODEL_REGISTRY[TaskComplexity.COMPLEX]
    total = 0.0

    for r in results:
        total += cfg.cost(r['in_tokens'], r['out_tokens'])

    return total


# -----------------------------
# Demo Tasks
# -----------------------------
tasks = [
    (
        'Classify a support ticket as billing, technical, or general',
        'Classify: "I was charged twice for my subscription"',
        'Return one word: billing, technical, or general'
    ),
    (
        'Summarise a 500-word complaint into 2 sentences',
        'Customer has repeated login failures for 3 weeks and contacted support 4 times.',
        'You are a concise summariser.'
    ),
    (
        'Analyse logs and propose a fix with code',
        '500 errors spike every 15 min, DB pool exhaustion observed.',
        'You are a senior backend engineer.'
    ),
]


# -----------------------------
# Run Demo
# -----------------------------
print('=== Dynamic Model Router Demo ===\n')

results = []
total_cost = 0.0

for task_desc, prompt, system in tasks:
    r = routed_call(task_desc, prompt, system)
    results.append(r)
    total_cost += r['cost_usd']

    print(f'Task    : {task_desc[:65]}')
    print(f'Routed  : complexity={r["complexity"].value:7s}  model={r["model"].model_id}')
    print(f'Tokens  : in={r["in_tokens"]}  out={r["out_tokens"]}  '
          f'latency={r["latency_s"]}s  cost=${r["cost_usd"]:.6f}')
    print(f'Response: {r["response"][:80]}...\n')


# -----------------------------
# Accurate Comparison
# -----------------------------
all_4o_cost = compute_all_gpt4o_cost(results)

print('--- Cost Summary ---')
print(f'Routed total cost : ${total_cost:.6f}')
print(f'All gpt-4o cost   : ${all_4o_cost:.6f}')

if all_4o_cost > total_cost:
    savings = all_4o_cost - total_cost
    print(f'✅ Savings         : ${savings:.6f}')
else:
    extra = total_cost - all_4o_cost
    print(f'⚠️ Routing overhead: ${extra:.6f}')

=== Dynamic Model Router Demo ===

Task    : Classify a support ticket as billing, technical, or general
Routed  : complexity=simple   model=gpt-4o-mini
Tokens  : in=33  out=1  latency=1.67s  cost=$0.000006
Response: billing...

Task    : Summarise a 500-word complaint into 2 sentences
Routed  : complexity=medium   model=gpt-4o-mini
Tokens  : in=34  out=20  latency=1.18s  cost=$0.000017
Response: The customer has experienced login issues for three weeks and has reached out to...

Task    : Analyse logs and propose a fix with code
Routed  : complexity=complex  model=gpt-4o
Tokens  : in=31  out=520  latency=7.09s  cost=$0.005278
Response: A spike in 500 errors every 15 minutes, coupled with database pool exhaustion, s...

--- Cost Summary ---
Routed total cost : $0.005300
All gpt-4o cost   : $0.005655
✅ Savings         : $0.000355


## 3b — Contextual Benchmarking & Interchangeability

In [50]:
# Benchmark models on YOUR task, not global leaderboards.
# Interchangeability: swap MODEL_REGISTRY above to change models everywhere.

def benchmark_models(test_cases: list[PromptTestCase],
                     models: list[str]) -> dict[str, dict]:
    results = {}
    for model_id in models:
        scores, costs, latencies = [], [], []
        for tc in test_cases:
            pv     = registry.get('classify_ticket', version='1.2.0')
            prompt = pv.render(ticket_text=tc.input)
            t0     = time.time()
            try:
                raw, in_toks, out_toks = call_openai(prompt, system=pv.system, model=model_id)
                latency = time.time() - t0
                clean   = re.sub(r'```json\s*|```', '', raw).strip()
                data    = json.loads(clean)
                out     = TicketClassification(**data)
                passed  = (out.category.value == tc.expected_category and
                           out.urgency.value  == tc.expected_urgency)
                scores.append(1 if passed else 0)
                # gpt-4o-mini pricing
                cost_1m_in  = 2.50 if 'gpt-4o' in model_id and 'mini' not in model_id else 0.15
                cost_1m_out = 10.0 if 'gpt-4o' in model_id and 'mini' not in model_id else 0.60
                costs.append((in_toks/1e6)*cost_1m_in + (out_toks/1e6)*cost_1m_out)
                latencies.append(latency)
            except Exception:
                scores.append(0)
                latencies.append(time.time() - t0)
                costs.append(0)

        n = len(test_cases)
        results[model_id] = {
            'accuracy':    sum(scores) / n,
            'avg_latency': sum(latencies) / n,
            'avg_cost':    sum(costs) / n,
            'total_cost':  sum(costs),
        }
    return results


print('=== Contextual Benchmark: your task, your models ===\n')
bench = benchmark_models(TEST_SUITE, models=['gpt-4o-mini', 'gpt-4o'])

print(f'{"Model":<20} {"Accuracy":>10} {"Avg Latency":>14} {"Avg Cost":>12} {"Total Cost":>12}')
print('-' * 72)
for model_id, m in bench.items():
    print(f'{model_id:<20} {m["accuracy"]:>9.0%}  {m["avg_latency"]:>11.2f}s'
          f'  {m["avg_cost"]:>11.5f}  {m["total_cost"]:>11.5f}')

# Show interchangeability: swap the MEDIUM model with a single config line
print('\n=== Interchangeability Demo ===')
print('Before:', MODEL_REGISTRY[TaskComplexity.MEDIUM].model_id)
MODEL_REGISTRY[TaskComplexity.MEDIUM] = ModelConfig(
    'gpt-4o', max_tokens=512, temperature=0.1,
    cost_per_1m_in=2.50, cost_per_1m_out=10.00
)
print('After :', MODEL_REGISTRY[TaskComplexity.MEDIUM].model_id)
print('All medium-complexity calls now use gpt-4o -- zero code changes elsewhere. ✅')


=== Contextual Benchmark: your task, your models ===

Model                  Accuracy    Avg Latency     Avg Cost   Total Cost
------------------------------------------------------------------------
gpt-4o-mini                71%         1.03s      0.00003      0.00024
gpt-4o                     86%         1.46s      0.00062      0.00437

=== Interchangeability Demo ===
Before: gpt-4o
After : gpt-4o
All medium-complexity calls now use gpt-4o -- zero code changes elsewhere. ✅


---
# Section 5 — Full Production Pipeline

Combines all three pillars into a single `ProductionTicketPipeline`:

```
Ticket arrives
    -> Pillar 3: routed_call() picks model by complexity
    -> Pillar 1: versioned prompt + Pydantic schema enforcement
    -> Pillar 2: BoundedContext.write() + ImmutableEventLog
    -> Audit trail + replay available for any ticket
```

In [ ]:
def safe_parse_json(raw: str) -> dict:
    import json, re

    # Remove markdown fences
    text = re.sub(r'```json\s*|```', '', raw).strip()

    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to extract JSON object
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    raise ValueError(f'Invalid JSON from model: {raw}')


class ProductionTicketPipeline:
    def __init__(self, prompt_registry: PromptRegistry,
                 event_log: ImmutableEventLog,
                 classify_ctx: BoundedContext,
                 assign_ctx:   BoundedContext,
                 resolve_ctx:  BoundedContext):
        self.registry     = prompt_registry
        self.event_log    = event_log
        self.classify_ctx = classify_ctx
        self.assign_ctx   = assign_ctx
        self.resolve_ctx  = resolve_ctx
        self.metrics: list[dict] = []

    def process(self, ticket_text: str) -> dict:
        tid = f'TKT-{uuid.uuid4().hex[:6].upper()}'

        # 1. Log receipt
        e_recv = Event(event_id=uuid.uuid4().hex[:8],
                       event_type=EventType.TICKET_RECEIVED,
                       entity_id=tid, payload={'text': ticket_text})
        self.event_log.append(e_recv)

        # 2. Classify (Pillar 3: routed, Pillar 1: versioned + schema)
        pv     = self.registry.get('classify_ticket', version='1.2.0')
        complexity = classify_complexity('classify support ticket')
        cfg    = MODEL_REGISTRY[complexity]
        prompt = pv.render(ticket_text=ticket_text)
        t0     = time.time()

        raw, in_toks, out_toks = call_openai(prompt, system=pv.system, model=cfg.model_id)
        latency = time.time() - t0
        data = safe_parse_json(raw)
        clf  = TicketClassification(**data)

        # 3. Write classification to its bounded context (Pillar 2)
        e_clf = self.classify_ctx.write(
            tid,
            {'category': clf.category.value, 'urgency': clf.urgency.value,
             'confidence': clf.confidence},
            EventType.TICKET_CLASSIFIED, caused_by=e_recv.event_id
        )

        # 4. Assign agent
        agent_map = {'technical': 'eng-team', 'billing': 'finance-team', 'general': 'support-team'}
        agent     = agent_map[clf.category.value]
        e_asgn    = self.assign_ctx.write(
            tid,
            {'assigned_to': agent, 'assigned_at': datetime.now(timezone.utc).isoformat()},
            EventType.AGENT_ASSIGNED, caused_by=e_clf.event_id
        )

        cost = cfg.cost(in_toks, out_toks)
        result = {'ticket_id': tid, 'category': clf.category.value,
                  'urgency': clf.urgency.value, 'confidence': clf.confidence,
                  'assigned_to': agent, 'model': cfg.model_id,
                  'latency_s': round(latency, 2), 'cost_usd': round(cost, 6)}
        self.metrics.append(result)
        return result

    def summary(self):
        n    = len(self.metrics)
        cost = sum(m['cost_usd'] for m in self.metrics)
        lat  = sum(m['latency_s'] for m in self.metrics) / n
        print(f'\n{"="*58}')
        print('PIPELINE SUMMARY')
        print(f'{"="*58}')
        print(f'Tickets processed : {n}')
        print(f'Avg latency       : {lat:.2f}s')
        print(f'Total cost        : ${cost:.5f}')
        print(f'Cost per ticket   : ${cost/n:.5f}')
        by_cat = {}
        for m in self.metrics:
            by_cat.setdefault(m['category'], []).append(m)
        print('\nBy category:')
        for cat, items in by_cat.items():
            print(f'  {cat:<12}: {len(items)} tickets  '
                  f'avg_confidence={sum(i["confidence"] for i in items)/len(items):.2f}')


# Fresh event log + contexts for the pipeline
fresh_log  = ImmutableEventLog()
clf_ctx    = BoundedContext('ClassificationService', {'category','urgency','confidence'}, fresh_log)
asgn_ctx   = BoundedContext('AssignmentService',     {'assigned_to','assigned_at'},       fresh_log)
res_ctx    = BoundedContext('ResolutionService',     {'status','resolved_at'},             fresh_log)

pipeline = ProductionTicketPipeline(registry, fresh_log, clf_ctx, asgn_ctx, res_ctx)

test_tickets = [
    'I was charged twice for my Pro subscription this month.',
    'Getting 500 errors on the /v2/embed endpoint since the last deploy.',
    'How do I export my data as CSV?',
    'My team cannot log in -- says account suspended but we paid.',
    'The dashboard charts are broken after the UI update.',
]

print('=== Full Production Pipeline Demo ===\n')
for t in test_tickets:
    r = pipeline.process(t)
    print(f'[{r["ticket_id"]}] {r["category"]:10s} | {r["urgency"]:6s} '
          f'| conf={r["confidence"]:.2f} | -> {r["assigned_to"]}'
          f'  [{r["model"]} {r["latency_s"]}s ${r["cost_usd"]:.5f}]')

pipeline.summary()

# Replay state for first ticket
first_id = pipeline.metrics[0]['ticket_id']
print(f'\nReplayed state for {first_id}:')
print(json.dumps(fresh_log.replay(first_id), indent=2))


=== Full Production Pipeline Demo ===

  LOG [ticket_received          ] entity=TKT-697BAB  payload={"text": "I was charged twice for my Pro subscription this m
prompt 
 Classify the following support ticket.

Allowed categories: "billing", "technical", "general"

Return JSON only using this schema:
Schema: {"category": "billing" | "technical" | "general", "urgency": "low" | "medium" | "high", "confidence": float (0.0 to 1.0)}

ßßßßRules:
- category must be one of the allowed values
- urgency reflects how critical the issue is
- confidence reflects classification certainty
- output strictly valid JSON (no markdown, no explanation)

Ticket: I was charged twice for my Pro subscription this month.
system 
 You are a precise support ticket classifier.
Classify tickets accurately into categories and assess urgency.
Always return valid JSON. Do not include any extra text.
model 
 gpt-4o-mini
raw 
 {"category":"billing","urgency":"high","confidence":0.95}
  LOG [ticket_classified        ] ent